In [ ]:
import os
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error
import kagglehub
from tqdm import tqdm
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score
from sklearn.model_selection import (
    train_test_split,
    KFold,
    StratifiedKFold,
    cross_val_score
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)
from sklearn.ensemble import (
    RandomForestRegressor,
    RandomForestClassifier
)

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
# 1. What does our target variable (delivery_time) look like?
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
df_original=df.copy() #copy the original df to clean the main one
# Task 1: Write your code here:
df=df.drop(columns=['Order_ID'])

In [ ]:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
# Task 2: Write your code here:
#we drop columns where delivery time is unkown becuase it is the target
df.dropna(subset="Delivery_Time",inplace=True)
#we make the unknown courier experience the mean since it follows a normal dist
df['Courier_Experience_yrs']=df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].mean())
#we make the categorical to unknown Since we can't infer them
df['Weather']=df['Weather'].fillna('unknown')
df['Time_of_Day']=df['Time_of_Day'].fillna('unknown')
df['Traffic_Level']=df['Traffic_Level'].fillna('unknown')

# df['Traffic_Level'].value_counts() used value counts to check for categories first

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)
df.info()

In [ ]:
# Task 4: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))
label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le
print("Label Encoded!")
df.head(-5)

In [ ]:
# Task 5: Write your code here:
#here we have to make sure we do not scale target so it does not affect the training and generate undesired scaling
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()


In [ ]:
# Task 6: Write your code here:
#not needed since it is not categorical

In [ ]:
# Task 1: Write your code here:
#  split

X = df.drop("Delivery_Time",axis=1)
y = df['Delivery_Time']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Delivery in train: {y_train.sum()}, in test: {y_test.sum()}")

In [ ]:

model= RandomForestRegressor(n_estimators=200)

# Storage for results
all_results = {}

all_results = {'mse': [], 'rmse': [], 'r2': [], 'mae':[]}

In [ ]:
#task2
kf = KFold(n_splits=5, shuffle=True, random_state=42)


for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{5}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]


  print(f"Training Forest Regressor...")

  # Task3: Train
  model.fit(X_train, y_train)

    # Predict
  y_pred = model.predict(X_test)

    # Calculate metrics
  mse = sklearn_mse(y_test, y_pred)
  rmse = np.sqrt(mse)
  r2 = r2_score(y_test, y_pred)
  mae=mean_absolute_error(y_test, y_pred)

    # Store results
  all_results["mse"].append(mse)
  all_results["rmse"].append(rmse)
  all_results["r2"].append(r2)
  all_results['mae'].append(mae)

In [ ]:
#task 4
print(f"\nForest Regressor:")
print(f"  MAE:  {np.mean(all_results['mae']):.4f}")



In [ ]:
# Task 1: Write your code here:
coeffs = {}

coeffs['Lasso'] = model.coef_
coeffs['Ridge'] = model.coef_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: